# Image Downloader

Handles downloading all product images from the provided URLs.
We'll use the `download_images` utility from `src/utils.py` to fetch images.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

from utils import download_images

## Load Dataset Files

We'll read both training and test CSVs to get all image links that need downloading.

In [ ]:
TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'

# Load datasets
print("Loading training data...")
train_df = pd.read_csv(TRAIN_PATH)
print(f"Training samples: {len(train_df)}")

print("\nLoading test data...")
test_df = pd.read_csv(TEST_PATH)
print(f"Test samples: {len(test_df)}")

print("\nDataset columns:")
print(train_df.columns.tolist())

Loading training data...
Training samples: 75000

Loading test data...
Test samples: 75000

Dataset columns:
['sample_id', 'catalog_content', 'image_link', 'price']


## Check Image Link Quality

Before downloading, let's examine the image_link column for any issues.

In [ ]:
# Combine all image links from train and test
all_image_links = pd.concat([
    train_df['image_link'],
    test_df['image_link']
], ignore_index=True)

print(f"Total image links (combined): {len(all_image_links)}")
print(f"Unique image links: {all_image_links.nunique()}")
print(f"Missing image links: {all_image_links.isna().sum()}")

# Show sample links
print("\nSample image URLs:")
for i, link in enumerate(all_image_links.dropna().head(3)):
    print(f"{i+1}. {link}")

Total image links (combined): 150000
Unique image links: 140587
Missing image links: 0

Sample image URLs:
1. https://m.media-amazon.com/images/I/51mo8htwTHL.jpg
2. https://m.media-amazon.com/images/I/71YtriIHAAL.jpg
3. https://m.media-amazon.com/images/I/51+PFEe-w-L.jpg


## Download Training Images

Download all images from the training set to a dedicated folder.
The utility uses multiprocessing for efficiency and handles retries automatically.

In [ ]:
# Create folder for training images
TRAIN_IMG_DIR = 'train_images'
Path(TRAIN_IMG_DIR).mkdir(exist_ok=True, parents=True)

print(f"Downloading {len(train_df)} training images to: {TRAIN_IMG_DIR}")

# Extract valid image links from training data
train_links = train_df['image_link'].dropna().tolist()

# Download using the provided utility function
download_images(train_links, str(TRAIN_IMG_DIR))

print("\nTraining image download complete!")

 52%|█████▏    | 38884/75000 [05:08<03:32, 169.78it/s]

HTTP Error 404: Not Found

 52%|█████▏    | 38918/75000 [05:08<03:31, 170.74it/s]

100%|██████████| 75000/75000 [09:32<00:00, 131.03it/s]



Training image download complete!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ML_Challenge2025/images_train/
!rsync -av --progress /content/train_images/ /content/drive/MyDrive/ML_Challenge2025/images_train/


Streaming output truncated to the last 5000 lines.
        571,422 100%    2.56MB/s    0:00:00 (xfr#69789, to-chk=2498/72288)
91VZJtP3tLL.jpg
        691,656 100%    2.20MB/s    0:00:00 (xfr#69790, to-chk=2497/72288)
91Va08HrE-L.jpg
        663,446 100%  829.57kB/s    0:00:00 (xfr#69791, to-chk=2496/72288)
91Vcx-b3iAL.jpg
        543,348 100%  651.86kB/s    0:00:00 (xfr#69792, to-chk=2495/72288)
91Vd+RlKrML.jpg
        637,676 100%  734.35kB/s    0:00:00 (xfr#69793, to-chk=2494/72288)
91VfcL0Hq7L.jpg
        806,580 100%  901.23kB/s    0:00:00 (xfr#69794, to-chk=2493/72288)
91VftAx9jrL.jpg
        866,122 100%  942.95kB/s    0:00:00 (xfr#69795, to-chk=2492/72288)
91Vg5I7jeGL.jpg
        554,815 100%  588.93kB/s    0:00:00 (xfr#69796, to-chk=2491/72288)
91VgFoeFeDL.jpg
        540,061 100%  553.41kB/s    0:00:00 (xfr#69797, to-chk=2490/72288)
91ViGdf9siL.jpg
        667,873 100%  665.53kB/s    0:00:00 (xfr#69798, to-chk=2489/72288)
91ViLxEMqGL.jpg
        572,889 100%  552.83kB/s    0:0

## Download Test Images

Similarly, download all test set images.

In [ ]:
# Create folder for test images
TEST_IMG_DIR = 'test_images'
Path(TEST_IMG_DIR).mkdir(exist_ok=True, parents=True)

print(f"Downloading {len(test_df)} test images to: {TEST_IMG_DIR}")

# Extract valid image links from test data
test_links = test_df['image_link'].dropna().tolist()

# Download using the provided utility function
download_images(test_links, str(TEST_IMG_DIR))

print("\nTest image download complete!")

 56%|█████▌    | 41914/75000 [05:22<03:41, 149.58it/s]

HTTP Error 404: Not Found


100%|██████████| 75000/75000 [09:17<00:00, 134.53it/s]



Test image download complete!


## Verify Downloads

Check how many images were successfully downloaded vs expected.

In [ ]:
# Count downloaded files
train_downloaded = len(list(Path(TRAIN_IMG_DIR).glob('*.jpg')))
test_downloaded = len(list(Path(TEST_IMG_DIR).glob('*.jpg')))

print("Download Summary:")
print("=" * 50)
print(f"Training images expected: {len(train_links)}")
print(f"Training images downloaded: {train_downloaded}")
print(f"Training success rate: {train_downloaded/len(train_links)*100:.2f}%")
print()
print(f"Test images expected: {len(test_links)}")
print(f"Test images downloaded: {test_downloaded}")
print(f"Test success rate: {test_downloaded/len(test_links)*100:.2f}%")
print("=" * 50)

# Note: Some images may fail to download due to network issues or broken links
# We'll handle missing images in the feature extraction phase

Download Summary:
Training images expected: 75000
Training images downloaded: 72287
Training success rate: 96.38%

Test images expected: 75000
Test images downloaded: 72221
Test success rate: 96.29%


## Create Image Mapping

Save a mapping of sample_id to image filename for easy lookup during feature extraction.

In [ ]:
# Function to extract filename from image link
def extract_image_filename(link):
    """Extract the filename portion from an image URL."""
    if pd.isna(link):
        return None
    return Path(link).name

# Create mappings for both datasets
train_df['image_filename'] = train_df['image_link'].apply(extract_image_filename)
test_df['image_filename'] = test_df['image_link'].apply(extract_image_filename)

train_mapping = train_df[['sample_id', 'image_filename', 'image_link']]
test_mapping = test_df[['sample_id', 'image_filename', 'image_link']]
train_mapping.to_csv('train_image_mapping.csv', index=False)
test_mapping.to_csv('test_image_mapping.csv', index=False)
print(f"\nSample train mapping:")
print(train_mapping.head())

Image mappings saved successfully!

Sample train mapping:
   sample_id   image_filename  \
0      33127  51mo8htwTHL.jpg   
1     198967  71YtriIHAAL.jpg   
2     261251  51+PFEe-w-L.jpg   
3      55858  41mu0HAToDL.jpg   
4     292686  41sA037+QvL.jpg   

                                          image_link  
0  https://m.media-amazon.com/images/I/51mo8htwTH...  
1  https://m.media-amazon.com/images/I/71YtriIHAA...  
2  https://m.media-amazon.com/images/I/51+PFEe-w-...  
3  https://m.media-amazon.com/images/I/41mu0HAToD...  
4  https://m.media-amazon.com/images/I/41sA037+Qv...  


## Summary

Images have been downloaded and organized for both training and test sets.
The mappings will help us link images back to their sample IDs during model training.

**Next Steps:**
- Exploratory Data Analysis (EDA) on catalog content and prices
- Text preprocessing and feature extraction
- Image feature extraction using CNNs